In [1]:
import os
import pandas as pd
import json
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math 
import random 
import igraph as ig
import networkx as nx
from math import log
import pickle
import gzip
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import copy
import community
import sys 
from pathlib import Path

import 

In [2]:

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[0]
SRC  = ROOT / "src"
DATA = ROOT / "data"
FIGS = ROOT / "figures"

print("Project root:", ROOT)
print("Data folder:", DATA)
print("Figures folder:", FIGS)


#import functions and libraries:
# Permette di importare moduli da src/
sys.path.insert(0, str(SRC))
from utils import * 
from label_colors import *
import label_colors

# === load your data ===
h = pickle.load(open(DATA / 'interaction_community_ex.pkl', 'rb'))
print("✅ Loaded community dictionary with", len(h), "entries")

csvdf2 = pd.read_csv(DATA / 'Subreddit_Tags.csv', sep=';', index_col=False)
Color_Field_df = pd.read_csv(DATA / 'Tag_Color.csv')

# ricrea i dizionari
Color_Field = dict(zip(Color_Field_df["Tag"], Color_Field_df["Color"]))

DizValTag = {} 

for i in range(len(csvdf2)) : 
    nome=csvdf2['subreddit'][i]
    n = 0
    for col in csvdf2.columns : 
        if col != 'subreddit' : 
            if csvdf2[col][i] != 0 : 
                n += 1 
    for col in csvdf2.columns : 
        if col != 'subreddit' : 
            if csvdf2[col][i] == 1 : 
                DizValTag[col+"_"+nome] = (1.0/n)
print(len(DizValTag)) 
print(DizValTag)


# === update the module's globals ===
label_colors.csvdf2 = csvdf2
label_colors.Color_Field = Color_Field
print("✅ Updated label_colors module globals")

Project root: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)
Data folder: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)/data
Figures folder: /Users/Daniele/Downloads/PolitoSphere Finale (Postzip)/figures
{'Geop': 'khaki', 'News': 'limegreen', 'Far-Left': 'cornflowerblue', 'Econ': 'darkturquoise', 'PolTalk': 'thistle', 'Politic': 'silver', 'Dem': 'navy', 'Lib': 'pink', 'Cons': 'crimson', 'Gun': 'darkred', 'Far-Right': 'blueviolet', 'Ban': 'black', 'SocialJustice': 'orangered', 'UK': 'orange', 'Eur': 'royalblue', 'Canada': 'gold', 'Austr': 'deeppink'}
✅ Loaded community dictionary with 7 entries
715
{'Dem_2012Elections': 0.5, 'Cons_2012Elections': 0.5, 'Dem_2016Elections': 0.5, 'Cons_2016Elections': 0.5, 'Dem_2016_elections': 0.5, 'Cons_2016_elections': 0.5, 'Lib_2ALiberals': 0.5, 'Gun_2ALiberals': 0.5, 'Politic_AOC': 0.5, 'Dem_AOC': 0.5, 'SocialJustice_Abortiondebate': 1.0, 'Geop_ActiveMeasures': 0.5, 'News_ActiveMeasures': 0.5, 'SocialJustice_AgainstHateSubreddits': 1.

Labels & Colors

In [3]:
community_labels = {}
community_colors = {}

for comm_id, nodes in h.items():
    label = label_colors.Label_from_list(nodes, DizValTag)
    color_rgb = label_colors.Color_from_list(nodes, DizValTag)
    color_hex = mcolors.to_hex(color_rgb)
    community_labels[comm_id] = label
    community_colors[comm_id] = color_hex
print(community_labels)
print(community_colors)

{0: 'UK/Eur', 1: 'Far-Left', 2: 'News/SocialJustice/Politic', 3: 'SocialJustice/Politic', 4: 'Cons/Lib/Politic', 5: 'Geop', 6: 'Canada'}
{0: '#c0914b', 1: '#6495ed', 2: '#82a234', 3: '#fb4c0b', 4: '#ea617c', 5: '#f0e68c', 6: '#ffd700'}


In [4]:
csvdf2

,subreddit,Geop,News,Far-Left,Econ,PolTalk,Politic,Dem,Lib,Cons,Gun,Far-Right,Ban,SocialJustice,UK,Eur,Canada,Austr
0,2012Elections,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0
1,2016Elections,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0
2,2016_elections,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0
3,2ALiberals,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0
4,AOC,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,willis7737_news,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
601,worldevents,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
602,worldpolitics,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
603,worldtoday,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Radar Plot

In [5]:
csvdf2 = pd.read_csv(DATA / 'Subreddit_Tags.csv', sep=';', index_col=False)
import utils
utils.csvdf2 = csvdf2

labels_2013 = list(community_labels.values())
colors_2013 = list(community_colors.values())

# ======================================================
# 🔹 Ordinamento e Radar Plot
# ======================================================

ordine = [
    "Geop", "News", "Far-Left", "Econ", "PolTalk", "Politic", "Dem", "Lib",
    "Cons", "Gun", "Far-Right", "Ban", "SocialJustice", "UK", "Eur", "Canada", "Austr"
]

Colori, Legende = [], []
Colori.append(colors_2013)
Legende.append(labels_2013)

for i, year in enumerate(range(2013, 2014)):
    labels_ord, colors_ord = ordina_legenda(Legende[i], Colori[i], ordine)
    crea_legenda_verticale_paper(
        labels=labels_ord,
        colors=colors_ord,
    )

values = []
for i in range(len(h)):
    values.append(Radar_from_list(h[i], DizValTag))

# ======================================================
# 🔹 ORDINA SECONDO LA LISTA DI PRIORITÀ
# ======================================================

labels_ord, colors_ord = ordina_legenda(labels_2013, colors_2013, ordine)

# Allinea anche i valori radar allo stesso ordine tematico
values_ord = [
    v for _, v in sorted(
        zip(labels_2013, values),
        key=lambda x: ordine.index(x[0].split("/")[0]) if x[0].split("/")[0] in ordine else len(ordine)
    )
]

titolo = "Community Radar Plot 2013"
plot_diagramma_radar_multiple(values_ord, colors_ord, labels_ord, titolo)
